<img src="logo-facty.jpeg" width="150" align="left"/>

<div style="text-align: center;">
    <h2><b>University of Carabobo</b></h2>
    <h3><b>Experimental Faculty of Sciences and Technology (FACYT)</b></h3>
    <h4><b>Computer Science Department</b></h4>
    <p style="text-align:center"><b>Course:</b> Machine Learning (Elective)<br>
    <b>Professor:</b> Álvaro Espinoza<br>
    <b>Date:</b> October 2026<br>
    <b>Students:</b> Juan Suárez and Josue Carrion</p>
</div>

<br clear="all">

---
# **Data Preparation and Feature Engineering**
## **StarCraft 2 Player Skill Prediction**

### **1. Problem Context and Purpose**
Before building the data preparation pipeline, it is necessary to establish the operational boundaries of the problem based on the Exploratory Data Analysis (EDA) findings:

*   **Unit of Observation:** A single player's performance metrics and telemetry recorded during a specific StarCraft 2 ranked match (`GameID`).
*   **Target Variable:** `LeagueIndex` (Integer ranging from 1 to 8, where 1 is Bronze and 8 is Professional). This frames our task as a Multiclass (and inherently Ordinal) Classification problem.
*   **Prediction Meaning:** A prediction represents the estimated competitive rank or skill tier of a player, derived strictly from their mechanical speed, attention shifting, and macro-economy management during a single game.
*   **Prediction Timestamp (Inference Moment):** The prediction occurs strictly **post-match**, immediately after a game concludes and the match replay file (`.SC2Replay`) is parsed by telemetry extraction tools. Therefore, all match-level aggregated telemetry (e.g., `APM`, `ActionLatency`, cumulative `TotalMapExplored`, `WorkersMade`) represents legitimate post-game descriptors fully available at inference time without temporal data leakage or lookahead bias.
*   **Dataset Limitation (Absence of PlayerID):** The dataset does not contain an explicit player identifier. While verifying that `GameID` is unique guarantees that match records are not duplicated, it does not prove independent and identically distributed (I.I.D.) observations across rows at the player level (a single player may have contributed multiple replays). This unavoidable constraint introduces a potential for intra-player correlation across splits (*group leakage*).
*   **Excluded Columns:** 
    *   `GameID`: Must be strictly excluded from all modeling pipelines. It is a surrogate key / unique identifier for the match carrying zero behavioral signal; keeping it would only invite spurious memorization or overfitting.

### **2. Connecting EDA Findings to Preparation Decisions**

Every preprocessing step and pipeline transformation in this notebook is strictly motivated by the empirical findings established during the Exploratory Data Analysis:

| EDA Finding | Empirical Evidence | Preparation Decision | Methodological & Domain Rationale |
| :--- | :--- | :--- | :--- |
| **Structured Missingness in League 8** | Exactly 100% of Professional players (League 8, n=55) lack demographic data (`Age`, `TotalHours`, `HoursPerWeek`), whereas amateur tiers show < 0.25% missing. | Apply `SimpleImputer(strategy='median')` strictly post-split **without** generating a missing indicator flag. | Generating a boolean indicator (`MissingIndicator`) or allowing tree models to split on NaNs would allow the classifier to exploit a **data collection artifact (shortcut learning)**, recognizing professionals solely from missingness rather than in-game telemetry. Imputation with the training median forces the algorithm to classify elite players based entirely on valid in-game mechanical metrics. |
| **Physically Impossible Demographics** | Outliers in demographic inputs: Max `TotalHours` = 1,000,000 (~114 years) and Max `HoursPerWeek` >= 168. | Apply deterministic filtering before the split (`TotalHours < 50000` and `HoursPerWeek < 168` while explicitly retaining NaNs). | Eliminates data entry errors that would otherwise destabilize imputation medians and scaling parameters, while preserving all 55 observations of League 8 intact. |
| **Valid Mechanical Ceilings** | Extreme values in telemetry: `APM` reaching up to 389.8 and `ActionLatency` down to 24 ms. | Retain all mechanical outliers; employ `RobustScaler` (median and IQR) for continuous scaling. | Extreme values reflect the legitimate physical ceiling and rapid keystroke mechanics of elite competitors, not errors. `RobustScaler` prevents these valid outliers from compressing the scale of middle-tier players. |
| **Pace-Reaction Interaction & Multicollinearity** | Marked inverse correlation ($r = -0.72$) between `APM` and `ActionLatency`. | Engineer `CognitiveSpeedIndex` (CSI = $\frac{\text{APM}}{\text{ActionLatency} + 1}$) as an experimental candidate feature. | Formulates a joint cognitive-motor efficiency ratio. Importantly, constructing this ratio does not automatically solve multicollinearity; retaining raw features alongside the ratio in linear models may increase collinearity. Its true predictive utility must be resolved via ablation benchmarking in the modeling phase. |
| **Target Class Imbalance** | Severe scarcity in extreme leagues (Bronze 1: 4.92%, GrandMaster 7: 1.03%, Professional 8: 1.62%), with majority in League 4 (23.90%). | Enforce **Stratified Sampling** (`stratify=y`) for the train/test split. | Stratification preserves proportional class representation in both partitions, preventing minority classes from disappearing. **Note:** Stratification preserves the imbalance; active mitigation (balanced class weights, macro metrics, resampling) is delegated strictly to within-fold cross-validation. |

### **3. Imports and Configuration**

We configure our execution environment with verified dependencies. To ensure clean traceability and pandas interoperability, we enable Scikit-Learn's native DataFrame output configuration (`transform_output="pandas"`).

In [1]:
# Standard Library
import os
import json
import warnings
import numpy as np
import pandas as pd

# Scikit-Learn: Global Configuration
from sklearn import set_config

# Scikit-Learn: Model Selection & Base
from sklearn.model_selection import train_test_split
from sklearn.base import clone

# Scikit-Learn: Pipeline and Transformers
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, FunctionTransformer
from sklearn.dummy import DummyClassifier

# Suppress minor warnings for clean output
warnings.filterwarnings('ignore')

# Global Scikit-Learn Configuration: output Pandas DataFrames instead of Numpy arrays
set_config(transform_output="pandas")

# --- Configuration Constants ---
RANDOM_SEED = 42
DATA_PATH = '../data/SkillCraft1_Dataset.csv'
TARGET_COL = 'LeagueIndex'
ID_COL = 'GameID'

# StarCraft 2 telemetry engine constant (1 timestamp = 1/88.5 seconds)
TIMESTAMPS_PER_MINUTE = 88.5 * 60

# Full list of expected raw feature columns from the dataset
REQUIRED_FEATURES = [
    'Age', 'HoursPerWeek', 'TotalHours', 'APM', 'SelectByHotkeys', 
    'AssignToHotkeys', 'UniqueHotkeys', 'MinimapAttacks', 
    'MinimapRightClicks', 'NumberOfPACs', 'GapBetweenPACs', 
    'ActionLatency', 'ActionsInPAC', 'TotalMapExplored', 
    'WorkersMade', 'UniqueUnitsMade', 'ComplexUnitsMade', 
    'ComplexAbilitiesUsed'
]

# Output directory for reproducible data artifacts
EXPORT_DIR = '../data'

print("[OK] Libraries successfully imported.")
print(f"[OK] Scikit-Learn transform_output configured to 'pandas'.")
print(f"[OK] Master random seed fixed at {RANDOM_SEED}.")

[OK] Libraries successfully imported.
[OK] Scikit-Learn transform_output configured to 'pandas'.
[OK] Master random seed fixed at 42.


### **4. Data Loading and Contract Validation**

In accordance with rigorous data engineering standards, we ingest the raw data and validate input contracts defensively. **Any structural violation of the contract (missing columns, duplicate primary keys, or invalid target categories) raises an immediate exception (`ValueError`) to halt the pipeline.**

Following contract verification, we execute the deterministic demographic filtering defined during the EDA.

In [2]:
# Ingest dataset treating '?' as missing values
df = pd.read_csv(DATA_PATH, na_values=['?'])

print("--- INPUT DATA CONTRACT VALIDATION ---")

# 1. Contract Check: Presence of Required Predictive Columns
missing_cols = [col for col in REQUIRED_FEATURES if col not in df.columns]
if len(missing_cols) == 0:
    print(f"[OK] All {len(REQUIRED_FEATURES)} required feature columns are present.")
else:
    raise ValueError(f"[CONTRACT VIOLATION] Missing required feature columns: {missing_cols}")

# 2. Contract Check: Presence of Target Variable
if TARGET_COL in df.columns:
    print(f"[OK] Target column '{TARGET_COL}' verified.")
else:
    raise ValueError(f"[CONTRACT VIOLATION] Target column '{TARGET_COL}' is missing from ingestion!")

# 3. Contract Check: Primary Key Uniqueness (GameID)
if ID_COL in df.columns:
    duplicates = df[ID_COL].duplicated().sum()
    if duplicates == 0:
        print(f"[OK] Primary key uniqueness verified: 0 duplicates in '{ID_COL}'.")
    else:
        raise ValueError(f"[CONTRACT VIOLATION] Found {duplicates} duplicate IDs in primary key '{ID_COL}'!")
else:
    raise ValueError(f"[CONTRACT VIOLATION] Primary key column '{ID_COL}' is missing!")

# 4. Contract Check: Valid Target Range (LeagueIndex must be strictly integer 1 to 8)
invalid_target_cats = df[~df[TARGET_COL].isin(range(1, 9))]
if len(invalid_target_cats) == 0:
    print(f"[OK] Target range verified: all values within valid competitive leagues [1 - 8].")
else:
    raise ValueError(f"[CONTRACT VIOLATION] Found {len(invalid_target_cats)} invalid categories in '{TARGET_COL}'! Values must be in range 1-8.")

print("\n--- DETERMINISTIC OUTLIER FILTERING (EDA Phase 1 Decisions) ---")
initial_shape = df.shape[0]

# Ensure demographic columns are properly cast as numeric
for col in ['Age', 'HoursPerWeek', 'TotalHours']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Filter physically impossible values while explicitly preserving NaNs to protect League 8
df = df[(df['TotalHours'] < 50000) | (df['TotalHours'].isna())]
df = df[(df['HoursPerWeek'] < 168) | (df['HoursPerWeek'].isna())]

filtered_shape = df.shape[0]
rows_dropped = initial_shape - filtered_shape

print(f"[ACTION] Deterministically filtered {rows_dropped} rows containing impossible demographic entries.")
print(f"[CHECK] Professional tier preserved intact: {df[df[TARGET_COL] == 8].shape[0]} observations in League 8.")
print(f"[INFO] Cleaned dataset ready for splitting: {df.shape[0]} rows × {df.shape[1]} columns.")

--- INPUT DATA CONTRACT VALIDATION ---
[OK] All 18 required feature columns are present.
[OK] Target column 'LeagueIndex' verified.
[OK] Primary key uniqueness verified: 0 duplicates in 'GameID'.
[OK] Target range verified: all values within valid competitive leagues [1 - 8].

--- DETERMINISTIC OUTLIER FILTERING (EDA Phase 1 Decisions) ---
[ACTION] Deterministically filtered 2 rows containing impossible demographic entries.
[CHECK] Professional tier preserved intact: 55 observations in League 8.
[INFO] Cleaned dataset ready for splitting: 3393 rows × 20 columns.


### **5. Train / Test Split and Leakage Prevention**

To rigorously prevent data leakage, the dataset is partitioned **before** any learned statistical transformations (imputations, medians, or scalers) are fitted.

*   **Stratified Sampling (`stratify=y`):** Given the severe class imbalance at the extremes (Leagues 1, 7, and 8), stratified sampling ensures that class proportions are preserved identically in both the training set (80%) and the test set (20%).
*   **Clarification on Class Imbalance:** Stratification ensures proportional representation, but **it does not balance the classes**. League 8 remains 1.62% of the data. Strategies to mitigate the impact of class imbalance (e.g., `class_weight='balanced'`, `Balanced Accuracy`, `Macro-F1`, or within-fold resampling) must be applied strictly during the modeling stage within each cross-validation training fold.
*   **Partition Disjunction vs. Subject Independence:** We verify that the training and test sets do not share any `GameID` records. However, because the dataset lacks an explicit `PlayerID`, this verifies disjunction at the match level only; true player-level independence cannot be mathematically proven.

In [3]:
print("--- TRAIN / TEST PARTITIONING ---")

# Separate predictive feature matrix (X) and target vector (y)
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Execute Stratified Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=RANDOM_SEED, 
    stratify=y
)

print(f"[OK] Stratified split completed successfully (Random Seed: {RANDOM_SEED}, Test Size: 20%).")
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

# 1. Verify disjunction of match identifiers
shared_games = len(set(X_train[ID_COL]).intersection(set(X_test[ID_COL])))
if shared_games == 0:
    print(f"\n[OK] Partition disjunction verified: 0 shared '{ID_COL}' records between Train and Test.")
    print(f"[NOTE] Disjunction confirmed at match level. In the absence of 'PlayerID', observations are assumed independent per match.")
else:
    raise ValueError(f"[DATA LEAKAGE ERROR] {shared_games} overlapping records found between Train and Test sets!")

# 2. Verify Stratification Fidelity
train_proportions = y_train.value_counts(normalize=True).sort_index() * 100
test_proportions = y_test.value_counts(normalize=True).sort_index() * 100
strat_audit = pd.concat([train_proportions, test_proportions, (test_proportions - train_proportions)], axis=1)
strat_audit.columns = ['Train %', 'Test %', 'Difference %']

print("\n--- Stratification Fidelity Audit across Leagues ---")
display(strat_audit.round(2))

--- TRAIN / TEST PARTITIONING ---
[OK] Stratified split completed successfully (Random Seed: 42, Test Size: 20%).
X_train shape: (2714, 19) | y_train shape: (2714,)
X_test shape:  (679, 19)  | y_test shape:  (679,)

[OK] Partition disjunction verified: 0 shared 'GameID' records between Train and Test.
[NOTE] Disjunction confirmed at match level. In the absence of 'PlayerID', observations are assumed independent per match.

--- Stratification Fidelity Audit across Leagues ---


,Train %,Test %,Difference %
LeagueIndex,,,
1,4.94,4.86,-0.08
2,10.21,10.31,0.10
3,16.29,16.35,0.06
4,23.91,23.86,-0.05
5,23.73,23.71,-0.02
6,18.28,18.26,-0.01
7,1.03,1.03,-0.00
8,1.62,1.62,-0.00


### **6. Feature Engineering (Custom Stateless Transformer)**

Following domain insights from StarCraft 2 and the empirical correlations identified in the EDA, we construct engineered features spanning four distinct families: **Ratios, Aggregations, Booleans, and Logarithmic Transformations**.

**1. Strategic Variables (Domain Rationale):**
*   **`CognitiveSpeedIndex` (Ratio):** $\frac{\text{APM}}{\text{ActionLatency} + 1}$. Combines physical command frequency (APM) with cognitive reaction latency. It is formulated as a candidate interaction feature. In this preparation pipeline, raw APM and ActionLatency are retained alongside CSI; downstream ablation experiments in modeling will test whether using CSI alone, raw features alone, or all three combined optimizes generalized predictive accuracy.
*   **`MinimapUsage` (Aggregation):** `MinimapAttacks + MinimapRightClicks`. Aggregates tactical commands issued directly onto the minimap radar, serving as a domain proxy for global macro-awareness.
*   **`ComplexMechanicsRate` (Aggregation) & `UsesComplexUnits` (Boolean):** Sum of advanced unit production and special spellcaster ability activations. Captures intense micro-management skill.
*   **`WeeksOfPlay` (Ratio):** $\frac{\text{TotalHours}}{\text{HoursPerWeek}}$. **Methodological Note:** Formulated as an **approximate proxy of the hypothetical number of weeks required to accumulate `TotalHours` at the currently reported weekly pace**. We explicitly recognize that assuming historical weekly dedication remained constant across a player's tenure is an unverified assumption; hence, this feature is treated as an experimental hypothesis subject to ablation testing.

**2. Skewness Compression & Per-Minute Unit Conversion:**
*   Our EDA demonstrated that action rates recorded per game-engine timestamp (~1/88.5 seconds) yield minuscule numerical values (~$10^{-4}$), where a standard $\log(1 + x) \approx x$ transform is mathematically ineffective.
*   We resolve this domain-specific technical nuance by first converting action rates to **actions per minute** (multiplying by $88.5 \times 60$) before applying $\log(1 + x)$.
*   Raw versions of log-transformed features are dynamically dropped inside the transformer to prevent raw/log duplication.

In [4]:
print("--- FEATURE ENGINEERING PREPARATION ---")

# Define columns requiring logarithmic transformation
HOUR_COLS = ["TotalHours", "HoursPerWeek"]
RATE_COLS = ["MinimapAttacks", "MinimapRightClicks", "SelectByHotkeys", "ComplexAbilitiesUsed"]
LOG_COLS = HOUR_COLS + RATE_COLS

def engineer_features(X):
    """
    Stateless feature engineering function for StarCraft 2 telemetry.
    Applies domain ratios, aggregations, boolean indicators, and unit-corrected log transforms.
    Ensures zero state is learned, making it fully safe for out-of-fold inference.
    """
    X_out = X.copy()
    
    # 1. Ratios and Strategic Aggregations
    X_out['CognitiveSpeedIndex'] = X_out['APM'] / (X_out['ActionLatency'] + 1.0)
    X_out['MinimapUsage'] = X_out['MinimapAttacks'] + X_out['MinimapRightClicks']
    X_out['ComplexMechanicsRate'] = X_out['ComplexUnitsMade'] + X_out['ComplexAbilitiesUsed']
    
    # 2. Tenure Proxy (replacing 0 with NaN to prevent division by zero)
    X_out['WeeksOfPlay'] = X_out['TotalHours'] / X_out['HoursPerWeek'].replace(0, np.nan)
    
    # 3. Boolean High-Tier Micro Indicator
    X_out['UsesComplexUnits'] = (X_out['ComplexUnitsMade'] > 0).astype(int)
    
    # 4. Skewness Reduction via Unit Conversion & log1p
    for c in HOUR_COLS:
        X_out[f"log_{c}"] = np.log1p(X_out[c])
        
    for c in RATE_COLS:
        # Convert engine timestamps to actions per minute, then apply log1p
        X_out[f"log_{c}"] = np.log1p(X_out[c] * TIMESTAMPS_PER_MINUTE)
        
    X_out["log_WeeksOfPlay"] = np.log1p(X_out.pop("WeeksOfPlay"))
    
    # 5. Drop raw counterparts of log-transformed features to avoid direct duplicates
    X_out = X_out.drop(columns=LOG_COLS)
    
    # Replace any mathematical infinities with NaNs for safe downstream median imputation
    return X_out.replace([np.inf, -np.inf], np.nan)

# Wrap into a Scikit-Learn FunctionTransformer
feature_engineer_transformer = FunctionTransformer(func=engineer_features, validate=False)

print("[OK] Custom Stateless Feature Engineering FunctionTransformer instantiated.")

--- FEATURE ENGINEERING PREPARATION ---
[OK] Custom Stateless Feature Engineering FunctionTransformer instantiated.


### **7. Feature Grouping and Organization**

To build an auditable `ColumnTransformer`, we group our columns based on their analytical requirements:
*   **`EXCLUDED`:** Match identifier (`GameID`), dropped to prevent surrogate key memorization.
*   **`BINARY`:** Categorical indicator (`UsesComplexUnits`), passed through untransformed (0 or 1).
*   **`NUMERIC_WITH_MISSING`:** Demographic and tenure features containing missing values (`Age`, `log_TotalHours`, `log_HoursPerWeek`, `log_WeeksOfPlay`), requiring median imputation followed by robust scaling.
*   **`NUMERIC`:** Standard telemetry variables with zero missingness, routed directly to robust scaling.

In [5]:
print("--- FEATURE GROUPING AND ROUTING SETUP ---")

# Generate temporary dry-run preview to extract feature names post-engineering
X_fe_preview = feature_engineer_transformer.transform(X_train)

# 1. Excluded Columns
EXCLUDED = [ID_COL]

# 2. Binary Features (Untransformed passthrough)
BINARY = ["UsesComplexUnits"]

# 3. Numeric Features with Missing Values (Requiring Median Imputation)
NUMERIC_WITH_MISSING = ["Age", "log_TotalHours", "log_HoursPerWeek", "log_WeeksOfPlay"]

# 4. Standard Numeric Features (Telemetry without missingness)
NUMERIC = [c for c in X_fe_preview.columns if c not in EXCLUDED + BINARY + NUMERIC_WITH_MISSING]

print(f"Excluded columns           ({len(EXCLUDED):2d}): {EXCLUDED}")
print(f"Binary columns             ({len(BINARY):2d}): {BINARY}")
print(f"Numeric with missing       ({len(NUMERIC_WITH_MISSING):2d}): {NUMERIC_WITH_MISSING}")
print(f"Standard numeric columns   ({len(NUMERIC):2d}): {NUMERIC}")

# Security check: Ensure every single column is assigned exactly once
all_assigned = EXCLUDED + BINARY + NUMERIC_WITH_MISSING + NUMERIC
assert sorted(all_assigned) == sorted(X_fe_preview.columns), "[ERROR] Column assignment mismatch!"

print(f"\n[OK] All {len(all_assigned)} features assigned to mutually exclusive pipeline routes.")

--- FEATURE GROUPING AND ROUTING SETUP ---
Excluded columns           ( 1): ['GameID']
Binary columns             ( 1): ['UsesComplexUnits']
Numeric with missing       ( 4): ['Age', 'log_TotalHours', 'log_HoursPerWeek', 'log_WeeksOfPlay']
Standard numeric columns   (18): ['APM', 'AssignToHotkeys', 'UniqueHotkeys', 'NumberOfPACs', 'GapBetweenPACs', 'ActionLatency', 'ActionsInPAC', 'TotalMapExplored', 'WorkersMade', 'UniqueUnitsMade', 'ComplexUnitsMade', 'CognitiveSpeedIndex', 'MinimapUsage', 'ComplexMechanicsRate', 'log_MinimapAttacks', 'log_MinimapRightClicks', 'log_SelectByHotkeys', 'log_ComplexAbilitiesUsed']

[OK] All 24 features assigned to mutually exclusive pipeline routes.


### **8. Sub-pipelines and Master ColumnTransformer Integration**

We construct dedicated Scikit-Learn sub-pipelines for each feature group and integrate them into a master `ColumnTransformer` (`preprocessor`), followed by the full `prep_pipeline`:

1.  **`numeric_missing_pipe`**: Combines `SimpleImputer(strategy='median')` with `RobustScaler()`.
2.  **`numeric_pipe`**: Encapsulates `RobustScaler()` alone.
3.  **`ColumnTransformer` (`preprocessor`)**: Routes features according to their defined group, sets `remainder='drop'` to eliminate unexpected columns, and sets `verbose_feature_names_out=False` to preserve clean original column names.
4.  **`prep_pipeline`**: Chains the stateless `feature_engineering` transformer with `preprocessing`.

In [6]:
print("--- SUB-PIPELINES AND PREPROCESSOR INTEGRATION ---")

# 1. Pipeline for numeric features with missing values
numeric_missing_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# 2. Pipeline for standard numeric telemetry (no missingness)
numeric_pipe = Pipeline(steps=[
    ('scaler', RobustScaler())
])

# 3. ColumnTransformer (The Traffic Controller)
preprocessor = ColumnTransformer(
    transformers=[
        ('num_missing', numeric_missing_pipe, NUMERIC_WITH_MISSING),
        ('num', numeric_pipe, NUMERIC),
        ('bin', 'passthrough', BINARY),
        ('drop_id', 'drop', EXCLUDED)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

# 4. Master Data Preparation Pipeline (Unfitted Template)
prep_pipeline = Pipeline(steps=[
    ('feature_engineering', feature_engineer_transformer),
    ('preprocessing', preprocessor)
])

print("[OK] ColumnTransformer configured with explicit branch routing.")
print("[OK] Master preparation pipeline (prep_pipeline) integrated successfully.")

--- SUB-PIPELINES AND PREPROCESSOR INTEGRATION ---
[OK] ColumnTransformer configured with explicit branch routing.
[OK] Master preparation pipeline (prep_pipeline) integrated successfully.


### **9. Structural Verification and Smoke Test**

> [!CAUTION]
> **CRITICAL METHODOLOGICAL DISTINCTION: SMOKE TEST vs. MODELING INPUT**
> The execution below is strictly a **diagnostic smoke test** to verify that the pipeline architecture executes without exceptions, transforms shapes accurately, and resolves all missing values.
> 
> **DO NOT use the transformed object `X_train_prep` as the input to cross-validation during modeling.**
> If models were evaluated on `X_train_prep`, the imputation medians and scaling parameters would have been learned from the entirety of `X_train`, leaking information into each cross-validation validation fold.
> 
> For modeling, **raw features must enter an end-to-end pipeline** that re-fits the preparation steps inside each fold.

In [7]:
print("--- STRUCTURAL SMOKE TEST ---")

# Fit and transform exclusively on X_train for structural diagnostics
X_train_prep = prep_pipeline.fit_transform(X_train)

# Transform X_test using parameters learned from X_train
X_test_prep = prep_pipeline.transform(X_test)

# Structural assertions
print(f"Original X_train features: {X_train.shape[1]}")
print(f"Prepared X_train features: {X_train_prep.shape[1]}")
print(f"Prepared X_test features:  {X_test_prep.shape[1]}")

missing_train = X_train_prep.isna().sum().sum()
missing_test = X_test_prep.isna().sum().sum()

print(f"\nMissing values in diagnostic X_train_prep: {missing_train}")
print(f"Missing values in diagnostic X_test_prep:  {missing_test}")

# Defensive verification
assert missing_train == 0, "[ERROR] Pipeline left missing values in training data!"
assert missing_test == 0, "[ERROR] Pipeline left missing values in testing data!"
assert X_train_prep.shape[1] == X_test_prep.shape[1] == 23, "[ERROR] Output feature dimension mismatch!"

print("\n[OK] Smoke test passed: Pipeline architecture generates 23 features with zero missing values.")
print("\nDiagnostic Prepared Training Sample:")
display(X_train_prep.head(3))

--- STRUCTURAL SMOKE TEST ---
Original X_train features: 19
Prepared X_train features: 23
Prepared X_test features:  23

Missing values in diagnostic X_train_prep: 0
Missing values in diagnostic X_test_prep:  0

[OK] Smoke test passed: Pipeline architecture generates 23 features with zero missing values.

Diagnostic Prepared Training Sample:


,Age,log_TotalHours,log_HoursPerWeek,log_WeeksOfPlay,APM,AssignToHotkeys,UniqueHotkeys,NumberOfPACs,GapBetweenPACs,ActionLatency,...,UniqueUnitsMade,ComplexUnitsMade,CognitiveSpeedIndex,MinimapUsage,ComplexMechanicsRate,log_MinimapAttacks,log_MinimapRightClicks,log_SelectByHotkeys,log_ComplexAbilitiesUsed,UsesComplexUnits
348,1.4,-0.520557,0.316611,-0.817990,-1.062095,-0.346656,0.000000,-1.677192,2.732269,1.391888,...,-1.000000,0.0,-0.782570,-0.575370,-0.127061,-0.132721,-0.786432,-1.068556,-0.152498,0
278,0.6,0.343194,0.566003,-0.145645,0.639762,-0.035159,0.666667,0.289269,0.087256,-0.478278,...,0.000000,0.0,0.700650,0.060788,-0.127061,1.105699,-0.309451,0.063481,-0.152498,0
2331,1.0,0.386010,1.234476,-0.734547,-0.663364,0.342431,0.000000,-0.303134,-0.211655,0.655620,...,0.333333,0.0,-0.529016,-0.477240,0.335185,-0.112929,-0.584364,-0.455573,0.617547,0


### **10. End-to-End Model Pipeline Factory (`build_model_pipeline`)**

To guarantee zero data leakage during cross-validation, every candidate model must be bundled into an end-to-end pipeline that receives **raw training rows** and encapsulates the preparation steps.

Using `sklearn.base.clone()`, we ensure that each cross-validation fold or experiment instantiates a pristine, unfitted copy of the preparation pipeline. Parameters (such as demographic medians and scaling ranges) are estimated strictly from the training folds and applied out-of-sample to the validation folds.

In [8]:
def build_model_pipeline(estimator, preparation_pipeline=prep_pipeline):
    """
    Constructs an end-to-end, unfitted modeling pipeline combining data preparation and a classifier.
    
    Uses `sklearn.base.clone()` to guarantee that the preparation pipeline holds no fitted state,
    preventing any preprocessing leakage across cross-validation folds.
    
    Parameters:
    -----------
    estimator : scikit-learn compatible classifier
    preparation_pipeline : Pipeline, default=prep_pipeline
    
    Returns:
    --------
    sklearn.pipeline.Pipeline
        A composite pipeline with steps:
        - 'preparation': Cloned, unfitted data preparation pipeline
        - 'model': The specified estimator
    """
    return Pipeline(steps=[
        ('preparation', clone(preparation_pipeline)),
        ('model', estimator)
    ])

print("--- MODEL PIPELINE FACTORY VERIFICATION ---")

# Smoke test the factory with a baseline DummyClassifier
baseline_dummy = DummyClassifier(strategy='most_frequent')
full_demo_pipeline = build_model_pipeline(baseline_dummy)

# Fit on raw X_train, y_train to prove end-to-end execution
full_demo_pipeline.fit(X_train, y_train)
demo_preds = full_demo_pipeline.predict(X_test)

print(f"[OK] Factory function 'build_model_pipeline' verified.")
print(f"[OK] Full pipeline successfully fit on raw X_train and generated {len(demo_preds)} test predictions.")
print(f"[OK] Cloned preparation pipeline maintains state isolation across experiments.")

--- MODEL PIPELINE FACTORY VERIFICATION ---
[OK] Factory function 'build_model_pipeline' verified.
[OK] Full pipeline successfully fit on raw X_train and generated 679 test predictions.
[OK] Cloned preparation pipeline maintains state isolation across experiments.


### **11. Feature Engineering Hypotheses and Ablation Study Plan**

Every engineered feature represents an empirical hypothesis. In the modeling phase, these hypotheses must be validated through a systematic **Ablation Study** under identical cross-validation folds (`StratifiedKFold(n_splits=5)`), evaluating models with `Balanced Accuracy` and `Macro-F1`:

| Experiment / Variant | Feature Configuration | Hypothesis / Analytical Purpose |
| :--- | :--- | :--- |
| **Variant 0: Raw Baseline** | Original 18 features (excluding `GameID`, dropping engineered features, standard robust scaling). | Evaluates baseline predictive capacity without any domain engineering. |
| **Variant 1A: Multicollinearity Test (Raw Only)** | `APM` + `ActionLatency` (excluding `CognitiveSpeedIndex`). | Benchmarks the predictive signal of raw pace and reaction latency. |
| **Variant 1B: Multicollinearity Test (CSI Only)** | `CognitiveSpeedIndex` (excluding raw `APM` and `ActionLatency`). | Tests whether condensing pace and latency into a single ratio resolves multicollinearity without degrading predictive performance. |
| **Variant 1C: Multicollinearity Test (Combined)** | `APM` + `ActionLatency` + `CognitiveSpeedIndex`. | Tests whether tree-based algorithms benefit from non-linear interaction terms despite linear collinearity. |
| **Variant 2: Tactical Aggregations Ablation** | Baseline + `MinimapUsage` + `ComplexMechanicsRate` + `UsesComplexUnits`. | Isolates the incremental value of macro-awareness and micro-management features. |
| **Variant 3: Tenure Proxy Ablation** | Baseline + `log_WeeksOfPlay`. | Tests whether the tenure proxy adds useful signal or behaves as uninformative noise. |
| **Variant 4: Full Feature Pipeline** | All 23 prepared features (current master pipeline). | Evaluates the overall composite feature set against all individual variants. |

### **12. Reproducible Handoff and Artifact Export**

To ensure full reproducibility independent of Jupyter notebook memory states, we export:
1.  **Raw Partitioned Datasets:** `X_train_raw.csv`, `X_test_raw.csv`, `y_train.csv`, `y_test.csv`.
2.  **Preparation Metadata Manifest:** `data_preparation_metadata.json` capturing seeds, dataset dimensions, target class proportions, and pipeline configuration.

In [9]:
print("--- EXPORTING REPRODUCIBLE DATA HANDOFF ARTIFACTS ---")

os.makedirs(EXPORT_DIR, exist_ok=True)

# 1. Export Raw Partition Datasets (Input to build_model_pipeline in notebook 03)
X_train_export_path = os.path.join(EXPORT_DIR, 'X_train_raw.csv')
X_test_export_path = os.path.join(EXPORT_DIR, 'X_test_raw.csv')
y_train_export_path = os.path.join(EXPORT_DIR, 'y_train.csv')
y_test_export_path = os.path.join(EXPORT_DIR, 'y_test.csv')

X_train.to_csv(X_train_export_path, index=False)
X_test.to_csv(X_test_export_path, index=False)
y_train.to_csv(y_train_export_path, index=False)
y_test.to_csv(y_test_export_path, index=False)

print(f"[EXPORT] Saved raw training features: {X_train_export_path} ({X_train.shape})")
print(f"[EXPORT] Saved raw testing features:  {X_test_export_path} ({X_test.shape})")
print(f"[EXPORT] Saved training target:      {y_train_export_path} ({y_train.shape})")
print(f"[EXPORT] Saved testing target:       {y_test_export_path} ({y_test.shape})")

# 2. Export Metadata Manifest
metadata_manifest = {
    "project": "StarCraft 2 League Index Predictor",
    "dataset": "SkillCraft1",
    "random_seed": RANDOM_SEED,
    "test_size": 0.20,
    "n_train_samples": int(X_train.shape[0]),
    "n_test_samples": int(X_test.shape[0]),
    "n_raw_features": int(X_train.shape[1]),
    "n_prepared_features": 23,
    "target_column": TARGET_COL,
    "target_classes": sorted(df[TARGET_COL].unique().tolist()),
    "league_8_train_proportion": float((y_train == 8).mean()),
    "league_8_test_proportion": float((y_test == 8).mean()),
    "naive_baseline_majority_class": int(y_train.mode()[0]),
    "naive_baseline_accuracy": float((y_train == y_train.mode()[0]).mean()),
    "excluded_features": EXCLUDED,
    "binary_features": BINARY,
    "numeric_with_missing": NUMERIC_WITH_MISSING,
    "standard_numeric": NUMERIC,
    "engineered_features": [
        "CognitiveSpeedIndex", "MinimapUsage", "ComplexMechanicsRate", 
        "UsesComplexUnits", "log_WeeksOfPlay", "log_TotalHours", 
        "log_HoursPerWeek", "log_MinimapAttacks", "log_MinimapRightClicks", 
        "log_SelectByHotkeys", "log_ComplexAbilitiesUsed"
    ],
    "handoff_contract": {
        "recommended_cv": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
        "pipeline_constructor": "build_model_pipeline(estimator)",
        "cv_input": "X_train_raw.csv (do NOT use pre-transformed data)"
    }
}

metadata_path = os.path.join(EXPORT_DIR, 'data_preparation_metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata_manifest, f, indent=4)

print(f"[EXPORT] Saved metadata manifest:    {metadata_path}")
print("\n[OK] Reproducible data handoff completed successfully.")

--- EXPORTING REPRODUCIBLE DATA HANDOFF ARTIFACTS ---
[EXPORT] Saved raw training features: ../data/X_train_raw.csv ((2714, 19))
[EXPORT] Saved raw testing features:  ../data/X_test_raw.csv ((679, 19))
[EXPORT] Saved training target:      ../data/y_train.csv ((2714,))
[EXPORT] Saved testing target:       ../data/y_test.csv ((679,))
[EXPORT] Saved metadata manifest:    ../data/data_preparation_metadata.json

[OK] Reproducible data handoff completed successfully.


### **13. Final Preparation Summary and Delivery Contract**

This notebook concludes the data preparation phase by establishing an auditable, leakage-free contract for the subsequent modeling phase:

**1. Data Preparation Strategy Implemented:**
*   **Leakage-Free Splitting:** Stratified 80/20 train/test split executed prior to any parameter estimation, preserving the 1.62% proportion of League 8 across partitions.
*   **Structured Missingness Resolution:** Demographic values imputed via training medians inside the pipeline without missing indicators, avoiding shortcut learning from collection artifacts.
*   **Variance Stabilization & Domain Scaling:** Action rates scaled from engine timestamps to actions per minute before `log1p` transformation, compressing extreme skewness.
*   **Robust Scaling:** Telemetry standardized using `RobustScaler` to protect valid mechanical ceilings of elite players.

**2. Delivery Contract to the Modeling Phase (`03_modeling.ipynb`):**
*   **Data Input:** Modeling must load `X_train_raw.csv` and `y_train.csv`. The test partition (`X_test_raw.csv`, `y_test.csv`) remains strictly locked in a vault until final model selection.
*   **Pipeline Encapsulation:** Every candidate algorithm must be constructed using `build_model_pipeline(estimator)` to ensure that all imputations and scalers are re-learned independently inside each cross-validation fold.
*   **Evaluation Protocol:** Cross-validation will be guided by `StratifiedKFold(n_splits=5)` using `Balanced Accuracy`, `Macro-F1`, and ordinal evaluation metrics, benchmarking against the **23.90%** naive baseline.
*   **Ablation Benchmarking:** The proposed feature variants (specifically the role of `CognitiveSpeedIndex` vs. raw `APM` and `ActionLatency`) will be compared out-of-sample before declaring a winning feature configuration.